This notebook is part of a video tutorial! See [here]() if you'd like to listen to an explanation of the code.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# FashionMNIST — 70,000 greyscale images of clothing items across 10 categories
# Each image is 28x28 pixels with 1 channel (greyscale)
raw_data  = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transforms.ToTensor())
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal',  'Shirt',   'Sneaker',  'Bag',   'Ankle boot']

print(f"Training images: {len(raw_data)}")
print(f"Test images    : {len(test_data)}")
print(f"Image shape    : {raw_data[0][0].shape}")   # (C, H, W)

# Level 1: Images Are Just Numbers
An image is a 3D tensor of shape **(channels × height × width)**.
Each value is a pixel intensity between 0 and 1.
Greyscale images have 1 channel; colour (RGB) images have 3.

In [ ]:
img, label = raw_data[0]

print(f"Label : {classes[label]}")
print(f"Shape : {img.shape}")                          # (1, 28, 28)
print(f"Values: min={img.min():.2f}, max={img.max():.2f}")

# img[0] strips the channel dimension — now it's just a 2D grid of pixel values
print(f"\nTop-left 5×5 corner (pixel values):\n{img[0, :5, :5]}")

In [ ]:
# Show one example of each class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('FashionMNIST — one sample per class', fontsize=14)

# Collect the first example we find for each class
seen = {}
for img_i, label_i in raw_data:
    if label_i not in seen:
        seen[label_i] = img_i
    if len(seen) == 10:
        break

for i, ax in enumerate(axes.flat):
    ax.imshow(seen[i].squeeze(), cmap='gray')   # squeeze() removes the channel dim for plotting
    ax.set_title(classes[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

### Exercise
Display the **first 12 images** from the training set in a 3×4 grid.
Label each subplot with the correct class name.

*Hint: you can unpack an image and label with `img_i, label_i = raw_data[i]`.*

# Level 2: Filters Detect Features
How does a model spot edges, curves, or textures in an image?

The core operation is **convolution**: slide a small filter (kernel) across the image, multiply element-wise with each patch, and sum the result. Each position produces one output value.

Let's implement it from scratch once — purely so you feel the pain and appreciate what comes next.

In [ ]:
# A vertical Sobel filter — highlights left/right edges
sobel_v = torch.tensor([
    [-1.,  0.,  1.],
    [-2.,  0.,  2.],
    [-1.,  0.,  1.]
])

img_2d = img.squeeze()         # (28, 28) — drop the channel dim
kH, kW = sobel_v.shape
H,  W  = img_2d.shape

# Slide the filter over every possible position — one patch at a time
output = torch.zeros(H - kH + 1, W - kW + 1)   # (26, 26)
for i in range(output.shape[0]):
    for j in range(output.shape[1]):
        patch      = img_2d[i:i+kH, j:j+kW]     # 3×3 region
        output[i, j] = (patch * sobel_v).sum()   # dot product

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img_2d, cmap='gray')
axes[0].set_title(f'Original: {classes[label]}')
axes[1].imshow(output.abs(), cmap='hot')
axes[1].set_title('Vertical edges (Sobel filter)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Exercise
A **blur filter** averages neighbouring pixels to smooth the image.
Replace `sobel_v` with the blur kernel below and run the loop again — what changes?

```python
blur = torch.ones(3, 3) / 9   # each of the 9 neighbours contributes equally
```

# Level 3: Build a CNN
Writing those loops for every filter, on every image, on every layer, would be unbearable.

`nn.Conv2d` handles the sliding window automatically — and crucially, the filter weights are **learnable parameters** that the network figures out from data.

Stack conv layers with activations and pooling and you have a CNN:
- **Conv2d** — applies a bank of learnable filters
- **ReLU** — sets negative values to zero (adds non-linearity so the model can learn complex patterns)
- **MaxPool2d** — halves the spatial size, making the model tolerant to small shifts
- **Linear** — maps the final features to class scores

In [ ]:
cnn = nn.Sequential(
    # Block 1 — learn low-level features (edges, curves)
    nn.Conv2d(1, 16, kernel_size=3, padding=1),   # (1,28,28) -> (16,28,28)
    nn.ReLU(),
    nn.MaxPool2d(2),                               # (16,28,28) -> (16,14,14)

    # Block 2 — combine low-level features into higher-level patterns
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # (16,14,14) -> (32,14,14)
    nn.ReLU(),
    nn.MaxPool2d(2),                               # (32,14,14) -> (32,7,7)

    # Classifier — flatten to a vector, then map to 10 class scores
    nn.Flatten(),                                  # (32,7,7) -> (1568,)
    nn.Linear(32 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

total_params = sum(p.numel() for p in cnn.parameters())
print(f"Total parameters: {total_params:,}")

# Test a forward pass with one image
sample = raw_data[0][0].unsqueeze(0)              # add batch dim: (1, 1, 28, 28)
logits = cnn(sample)
print(f"Output shape    : {logits.shape}")         # (1, 10) — one score per class
print(f"Predicted class : {classes[logits.argmax().item()]}  (weights are random, so this is meaningless for now)")

### Exercise
Add `nn.Dropout(p=0.3)` just before the final `nn.Linear(128, 10)`.

Dropout randomly zeroes 30% of activations during training, which forces the network to not rely too heavily on any one neuron — a simple trick that often improves generalisation.

# Level 4: Train It
Loading the full 60k training images at once would exhaust your RAM and give slow, noisy gradient updates.

`DataLoader` splits the data into **mini-batches** and shuffles them each epoch. `torch.optim` handles the gradient updates.

In [ ]:
train_loader = DataLoader(raw_data,  batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=64)

optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)

def train_one_epoch(model, loader, opt):
    model.train()      # turns on dropout, batch norm, etc.
    total_loss = 0
    for images, labels in loader:
        opt.zero_grad()
        loss = F.cross_entropy(model(images), labels)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def accuracy(model, loader):
    model.eval()       # turns off dropout, etc. for evaluation
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            correct += (model(images).argmax(dim=1) == labels).sum().item()
    return correct / len(loader.dataset)

for epoch in range(5):
    loss = train_one_epoch(cnn, train_loader, optimizer)
    acc  = accuracy(cnn, test_loader)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Test Accuracy: {acc:.1%}")

### Exercise
Swap `torch.optim.Adam` for SGD with momentum:
```python
optimizer = torch.optim.SGD(cnn.parameters(), lr=0.01, momentum=0.9)
```
Retrain from scratch (re-run the `cnn = nn.Sequential(...)` cell first to reset the weights).
Does the loss go down faster or slower?

# Level 5: Make the Data Work Harder
Your model only learns from what it sees. **Data augmentation** creates new variations of each training image on the fly.

Same image, random transformation → the model sees more variety without collecting more data.

In [ ]:
augment_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),              # randomly mirror left-right
    transforms.RandomAffine(degrees=10,             # rotate up to 10 degrees
                            translate=(0.1, 0.1)),  # shift up to 10% in any direction
    transforms.ToTensor(),
])

train_aug        = datasets.FashionMNIST(root='./data', train=True, download=False, transform=augment_transform)
train_loader_aug = DataLoader(train_aug, batch_size=64, shuffle=True)

# Fresh model — reset weights so we start from scratch
cnn2 = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
    nn.Linear(128, 10),
)
optimizer2 = torch.optim.Adam(cnn2.parameters(), lr=1e-3)

for epoch in range(5):
    loss = train_one_epoch(cnn2, train_loader_aug, optimizer2)
    acc  = accuracy(cnn2, test_loader)              # test_loader has no augmentation — fair comparison
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Test Accuracy: {acc:.1%}")

### Exercise
Add `transforms.RandomRotation(degrees=15)` to the augmentation pipeline.

Does it help or hurt accuracy? Think about why — is rotating a sandal by 15° still a realistic training example?

# Level 6: And Beyond
See [torchvision docs](https://pytorch.org/vision/stable/index.html) for datasets, transforms, and pretrained models.

See the [PyTorch CV tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) for a full end-to-end walkthrough on colour images.

In [ ]:
import torchvision.models as models

# ResNet-18 was pretrained on 1.2 million ImageNet images — it already knows a lot about visual features
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all layers — preserve what it already learned
for param in resnet.parameters():
    param.requires_grad = False

# Replace only the final classification layer for our 10 classes
resnet.fc = nn.Linear(resnet.fc.in_features, 10)   # only this layer will train

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet.parameters())
print(f"Training {trainable:,} of {total:,} parameters ({trainable/total:.1%})")
print("\nInstead of training 11 million parameters from scratch,")
print("we only need to train the final layer — much faster and often more accurate.")

In [ ]:
# Convert grayscale (1 channel) to RGB (3 channels) by repeating the channel
def convert_to_rgb(batch):
    images, labels = batch
    # Repeat single channel 3 times: (B, 1, H, W) -> (B, 3, H, W)
    images = images.repeat(1, 3, 1, 1)
    return images, labels

class RGBDataLoader:
    def __init__(self, loader):
        self.loader = loader
    
    def __iter__(self):
        for batch in self.loader:
            yield convert_to_rgb(batch)
    
    def __len__(self):
        return len(self.loader)

rgb_train_loader = RGBDataLoader(train_loader)
rgb_test_loader = RGBDataLoader(test_loader)

optimizer3 = torch.optim.Adam(resnet.parameters(), lr=1e-3)

for epoch in range(5):
    loss = train_one_epoch(resnet, rgb_train_loader, optimizer3)
    acc  = accuracy(resnet, rgb_test_loader)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Test Accuracy: {acc:.1%}")

I got AI to generate the visualisation code below.

In [ ]:
# Visualise predictions from the trained CNN on a batch of test images
images, labels = next(iter(DataLoader(test_data, batch_size=16, shuffle=True)))

cnn.eval()
with torch.no_grad():
    preds = cnn(images).argmax(dim=1)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('CNN Predictions on FashionMNIST Test Set', fontsize=16)

for idx, ax in enumerate(axes.flat):
    ax.imshow(images[idx].squeeze(), cmap='gray')
    correct = preds[idx].item() == labels[idx].item()
    colour  = 'green' if correct else 'red'
    ax.set_title(f"Pred: {classes[preds[idx]]}\nTrue: {classes[labels[idx]]}",
                 color=colour, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()